In [1]:
import random
from pathlib import Path

import torch
import torch.nn.functional as F

In [2]:
data_path = Path("parole.txt")
if not data_path.is_file():
    data_path = Path("ItalianWords") / "parole.txt"

words = data_path.read_text(encoding="utf-8").splitlines()

In [3]:
words[:10]

['a',
 'ab',
 'abaco',
 'abadessa',
 'abalietà',
 'abanese',
 'abarica',
 'abarico',
 'abasia',
 'abate']

In [4]:
padded_words = []

for w in words:
    word = "." + w + "."
    padded_words.append(word)

padded_words[:10]

['.a.',
 '.ab.',
 '.abaco.',
 '.abadessa.',
 '.abalietà.',
 '.abanese.',
 '.abarica.',
 '.abarico.',
 '.abasia.',
 '.abate.']

In [5]:
char = sorted(set("".join(words)))
print(len(char))

32


In [6]:
# Build the vocabulary of characters and mappings to/from integers.
chars = sorted(set("".join(words)))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi["."] = 0
itos = {i: s for s, i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 27: 'à', 28: 'è', 29: 'é', 30: 'ì', 31: 'ò', 32: 'ù', 0: '.'}


In [7]:
# Build the dataset.
block_size = 10  # Context length: how many characters do we take to predict the next one?


def build_dataset(words, verbose=False):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + ".":
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            if verbose:
                print("".join(itos[i] for i in context), "--->", itos[ix])
            context = context[1:] + [ix]  # Crop and append.

    X = torch.tensor(X, dtype=torch.long)
    Y = torch.tensor(Y, dtype=torch.long)
    print(X.shape, Y.shape)
    return X, Y


torch.manual_seed(42)
random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

torch.Size([764283, 10]) torch.Size([764283])
torch.Size([95706, 10]) torch.Size([95706])
torch.Size([95180, 10]) torch.Size([95180])


Now, let's create the net. Following Andrej, I want to create my own NN class so that I can understand how to move from his course-style notes to my style of reasoning through OOP.

In [8]:
class Embeddings:
    def __init__(self, n_inputs, emb_space):
        self.emb = torch.randn((n_inputs, emb_space), requires_grad=True)

    def __call__(self, indices):
        return self.emb[indices]

    def parameters(self):
        return [self.emb]

### Visualizing the embedding lookup

Following Andrej Karpathy's [Makemore](https://github.com/karpathy/makemore) examples, let's visualize what happens when we call our embedding object: `C(Xtr[:5])`. We select five input examples, each containing a sequence of ten character indices, so the input has shape `(5, 10)`. Each character index is replaced by a learned vector of two numbers. One character therefore becomes a vector of shape `(2,)`, and one ten-character context becomes a matrix of shape `(10, 2)`. Stacking the five contexts gives a three-dimensional tensor of shape `(5, 10, 2)`: **five examples, ten characters per example, two embedding coordinates per character**. The lookup preserves the order of the characters; repeated indices retrieve the same vector.

In [9]:
C = Embeddings(len(stoi), 2)
print(C(Xtr[:5]))
print(C(Xtr[:5]).shape)

tensor([[[ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873]],

        [[ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [-1.3847, -0.8712]],

        [[ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [-1.3847, -0.8712],
         [-0.3925, -1.4036]],

        [[ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
        

Now, let's define our Neural Neuron.

In [10]:
class Neuron:
    def __init__(self, n_inputs):
        self.w = torch.randn(n_inputs, requires_grad=True)
        self.b = torch.zeros((), requires_grad=True)

    def __call__(self, x, act=None):
        z = x @ self.w + self.b
        if act is None:
            return torch.tanh(z)
        return act(z)

    def parameters(self):
        return [self.w, self.b]

In [11]:
class Layer:
    def __init__(self, nin, nout, act=None):
        self.neurons = [Neuron(nin) for _ in range(nout)]
        self.act = act

    def __call__(self, x):
        outs = [neuron(x, act=self.act) for neuron in self.neurons]
        return torch.stack(outs, dim=-1)

    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]

In [12]:
# Example of torch.stack.
a = torch.tensor([1, 2, 3])
b = torch.tensor([4, 5, 6])
print(a, b)
print(torch.stack([a, b], dim=0))
print(torch.stack([a, b], dim=-1))

tensor([1, 2, 3]) tensor([4, 5, 6])
tensor([[1, 2, 3],
        [4, 5, 6]])
tensor([[1, 4],
        [2, 5],
        [3, 6]])


In [13]:
def identity(x):
    return x


class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [
            Layer(
                sz[i],
                sz[i + 1],
                act=identity if i == len(nouts) - 1 else None,
            )
            for i in range(len(nouts))
        ]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

OK, now we have almost everything that we need to make our first forward pass: the embeddings, a neuron, the layers and the MLP that combines the layers with the neurons.

We are still missing something, though. The embeddings need to be "preprocessed" before being fed to the MLP.

The easiest way possible, following Andrej's idea, is to "flatten" the embeddings in order to have just one vector for the input.

As an example, if we take the first 5 data points:

In [14]:
c = C(Xtr[:5])
print(c)

tensor([[[ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873]],

        [[ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [-1.3847, -0.8712]],

        [[ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [-1.3847, -0.8712],
         [-0.3925, -1.4036]],

        [[ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
        

We see that, sorry for the lack of formalism, the tensor is a "vector of size 5, and each entry is a matrix". What we want to make is a matrix with 5 rows and, in this case, 20 columns (embedding dimensions times the context length).

In PyTorch we can do this in 3 ways:

In [15]:
flatten1 = c.view(-1, block_size * C.emb.shape[1])
# Or
flatten2 = c.view(c.shape[0], -1)
# Or
flatten3 = c.flatten(start_dim=1)
print(flatten1)
print(flatten2)
print(flatten3)

tensor([[ 1.9269,  1.4873,  1.9269,  1.4873,  1.9269,  1.4873,  1.9269,  1.4873,
          1.9269,  1.4873,  1.9269,  1.4873,  1.9269,  1.4873,  1.9269,  1.4873,
          1.9269,  1.4873,  1.9269,  1.4873],
        [ 1.9269,  1.4873,  1.9269,  1.4873,  1.9269,  1.4873,  1.9269,  1.4873,
          1.9269,  1.4873,  1.9269,  1.4873,  1.9269,  1.4873,  1.9269,  1.4873,
          1.9269,  1.4873, -1.3847, -0.8712],
        [ 1.9269,  1.4873,  1.9269,  1.4873,  1.9269,  1.4873,  1.9269,  1.4873,
          1.9269,  1.4873,  1.9269,  1.4873,  1.9269,  1.4873,  1.9269,  1.4873,
         -1.3847, -0.8712, -0.3925, -1.4036],
        [ 1.9269,  1.4873,  1.9269,  1.4873,  1.9269,  1.4873,  1.9269,  1.4873,
          1.9269,  1.4873,  1.9269,  1.4873,  1.9269,  1.4873, -1.3847, -0.8712,
         -0.3925, -1.4036, -0.2316,  0.0418],
        [ 1.9269,  1.4873,  1.9269,  1.4873,  1.9269,  1.4873,  1.9269,  1.4873,
          1.9269,  1.4873,  1.9269,  1.4873, -1.3847, -0.8712, -0.3925, -1.4036,
      

The first one is saying: starting from c, create a 2D view by inferring the number of rows, given that we want 20 columns; here the vectors are row vectors.

The second is the reverse: it is saying to create a 2D view with the number of rows equal to the number of input points, here 5, and infer the second dimension.

The third is saying, instead, start flattening from dimension 1; the embedding tensor has shape (5, 10, 2). So it is saying to keep the first dimension, the 5 input data points, and then flatten all the others, making a (5, 20) matrix.

I will use this one for this purpose.

In [16]:
class LanguageModel:
    def __init__(self, n_vocab, n_emb, block_size, nouts):
        self.emb = Embeddings(n_vocab, n_emb)
        self.mlp = MLP(block_size * n_emb, nouts)
        self.last_output = None

    def forward(self, indices):
        x = self.emb(indices)
        x = x.flatten(start_dim=1)
        out = self.mlp(x)
        self.last_output = out
        return out

    def __call__(self, indices):
        return self.forward(indices)

    def parameters(self):
        return self.emb.parameters() + self.mlp.parameters()

    def NLL(self, y):
        if self.last_output is None:
            raise ValueError("Need to make a forward pass")
        loss = F.cross_entropy(self.last_output, y)
        return loss

    def fit(self, X, Y, steps, batch, lr):
        params = self.parameters()
        for step in range(steps):
            ix = torch.randint(len(X), (batch,))
            Xix, Yix = X[ix], Y[ix]
            logits = self(Xix)  # Thanks to the __call__ method.
            loss = F.cross_entropy(logits, Yix)

            # We clear the gradients at each step to calculate the gradient with respect to the newest loss.
            for p in params:
                p.grad = None

            loss.backward()

            with torch.no_grad():
                for p in params:
                    p -= lr * p.grad  # Always -= since we are going down the gradient (gradient descent).

            if step % 100 == 0:
                print(step, loss.item())

    def params_num(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

OK, let's do the first forward pass:

In [17]:
Model = LanguageModel(
    n_vocab=len(stoi),
    block_size=block_size,
    n_emb=2,
    nouts=[1000, len(stoi)],
)

result = Model(Xtr[:5])

In [18]:
print(Model.params_num())
print(result.shape)

54099
torch.Size([5, 33])


Following Andrej's interpretation, we can think of this as "log-counts".
We can now use the softmax function to create a probability distribution over the possible character outcomes:

In [19]:
result.softmax(dim=1).sum(dim=1)

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<SumBackward1>)

In [20]:
print(torch.arange(len(Ytr[:5])), Ytr[:5])

tensor([0, 1, 2, 3, 4]) tensor([16,  5, 14, 20,  1])


In [21]:
log_prob = torch.log_softmax(result, dim=1)
loss = -log_prob[torch.arange(len(Ytr[:5])), Ytr[:5]].mean()
loss

tensor(75.2923, grad_fn=<NegBackward0>)

In [22]:
Model.fit(Xtr, Ytr, 10000, batch=50, lr=0.01)

0 68.52084350585938


100 16.740297317504883


200 5.3638386726379395


300 4.008847236633301


400 3.4542298316955566


500 3.475048303604126


... (90 outputs left out)

9600 3.061208486557007


9700 2.6992228031158447


9800 2.7762715816497803


9900 2.910087823867798


In [23]:
# Generate 10 words using the trained model.
with torch.no_grad():
    for _ in range(10):
        context = []
        for i in range(block_size):
            context.append(0)

        generated = []

        for _ in range(50):  # Length limit.
            x = torch.tensor([context], dtype=torch.long)
            # Shape: (1, block_size), a single example.

            logits = Model(x)  # (1, len(stoi))
            probs = torch.softmax(logits, dim=1)

            ix = torch.multinomial(probs, num_samples=1).item()

            if ix == stoi["."]:
                break

            generated.append(itos[ix])
            context = context[1:] + [ix]

        print("".join(generated))

indisretaraca
pomiarioa
nelpeiihz
cqdesrgnto
inntdrietcnlle
uncatiata
mpliatrtiiebn
aaefllinsroms
dibeazo
irasrcslozerieeicolee
